# BLAST — busca e interpretação de similaridade

Disciplina: **EQM — Bioinformática e Biologia Molecular**

Nesta prática vamos recuperar a sequência `MK270576.1`, executar uma busca `blastn`
e interpretar identidade, cobertura, E-value e bit score.

> A busca remota usa os servidores do NCBI e pode levar alguns minutos.

## 1. Montar o Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
BASE = Path("/content/drive/MyDrive/Bioinformatica_Biologia_Molecular/02_blast")
BASE.mkdir(parents=True, exist_ok=True)
print(BASE)

## 2. Recuperar a sequência consulta em FASTA

In [ ]:
import urllib.request
accession = "MK270576.1"
url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id={accession}&rettype=fasta&retmode=text"
query = BASE / f"{accession}.fasta"
urllib.request.urlretrieve(url, query)
print(query.read_text()[:500])

## 3. Instalar BLAST+

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ncbi-blast+
!blastn -version

## 4. Executar BLASTn remoto

Usaremos uma saída tabular com:
`qseqid`, `sacc`, `% identity`, comprimento do alinhamento, query coverage,
E-value, bit score e título do hit.

In [ ]:
query_path = str(query)
saida = BASE / "blastn_nt.tsv"

!blastn   -query "$query_path"   -db nt   -remote   -max_target_seqs 15   -outfmt "6 qseqid sacc pident length qcovs evalue bitscore stitle"   -out "$saida"

print(saida)

## 5. Ler e organizar os resultados

In [ ]:
import pandas as pd

cols = ["query","accession_hit","identity_pct","alignment_length","query_coverage_pct","evalue","bitscore","title"]
df = pd.read_csv(saida, sep="\t", names=cols)

df.head(15)

## 6. Ordenar e inspecionar

In [ ]:
df.sort_values(["evalue","bitscore"], ascending=[True,False]).head(15)

## 7. Interpretação

Observe os três primeiros hits e responda:

- qual possui maior cobertura da query?
- qual possui maior identidade?
- os melhores valores correspondem ao mesmo organismo?
- o título do registro é suficiente para uma identificação definitiva?

In [ ]:
df[["accession_hit","identity_pct","query_coverage_pct","evalue","bitscore","title"]].head(5)

## 8. Exercício

Troque o accession por outra sequência da prática anterior e repita a busca.
Compare os resultados.

**Mensagem:** BLAST mede similaridade local. A interpretação taxonômica depende do
contexto, da qualidade e do conteúdo do banco, da cobertura e do tipo de marcador.